In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os

In [22]:
path = kagglehub.dataset_download("lakshmi25npathi/online-retail-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'online-retail-dataset' dataset.
Path to dataset files: /kaggle/input/online-retail-dataset


In [23]:
dataset_path = "/root/.cache/kagglehub/datasets/lakshmi25npathi/online-retail-dataset/versions/1"

files = os.listdir(dataset_path)
print(files)

['online_retail_II.xlsx']


In [24]:
def extract(dataset_path):

    file_path = dataset_path
    df = pd.read_excel(file_path)
    return df

In [25]:
def transform(df):

  #Handling Missing Value i.e removing duplicates and null
  df = df.dropna(subset=['Customer ID'])
  df.drop_duplicates(inplace=True)

  #Creating Features
  df['Revenue'] = df['Quantity'] * df['Price']
  df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
  df['TotalRevenue'] = df.groupby('Customer ID')['Revenue'].transform('sum')
  top_product_desc = df.groupby('Description')['Revenue'].sum().idxmax()
  df['Is_Most_Popular_Product'] = df['Description'] == top_product_desc
  df['OrderSize'] = df.groupby('Invoice')['Quantity'].transform('sum')

  #Applying Lamda Function
  rev_33 = df['TotalRevenue'].quantile(0.33)
  rev_67 = df['TotalRevenue'].quantile(0.67)
  df['CustomerTier'] = df['TotalRevenue'].apply(
        lambda x: 'Low' if x <= rev_33 else ('Medium' if x <= rev_67 else 'High')
    )

  df['Contains_Gift'] = df['Description'].apply(
        lambda x: True if 'GIFT' in str(x).upper() else False
    )

  trans_33 = df['Revenue'].quantile(0.33)
  trans_67 = df['Revenue'].quantile(0.67)
  df['TransactionSize'] = df['Revenue'].apply(
        lambda x: 'Small' if x <= trans_33 else ('Medium' if x <= trans_67 else 'Large')
    )

  df['Is_Christmas_Themed'] = df['Description'].apply(
        lambda x: True if 'CHRISTMAS' in str(x).upper() else False
    )

  invoice_counts = df.groupby('Customer ID')['Invoice'].transform('nunique')
  df['CustomerLoyalty'] = invoice_counts.apply(
        lambda x: 'One-time' if x == 1 else ('Occasional' if x <= 5 else 'Loyal')
    )

  unique_items_per_invoice = df.groupby('Invoice')['StockCode'].transform('nunique')
  df['MultiItem_Order'] = unique_items_per_invoice.apply(
        lambda x: True if x > 1 else False
    )

  return df



In [26]:
def load(df, output_path):
    #df.to_csv(output_path, index=False)
    print("Data saved")
    return df

def ETL(dataset_path, output_path):
    df = extract(dataset_path)
    df_transformed = transform(df)
    df_loaded = load(df_transformed, output_path)

    return df_loaded

In [27]:
dataset_path = "/root/.cache/kagglehub/datasets/lakshmi25npathi/online-retail-dataset/versions/1/online_retail_II.xlsx"
output_path = "/content/transformed_retail_data.csv"

df = ETL(dataset_path, output_path)
df.head()

/tmp/ipykernel_248/2037780541.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)
/tmp/ipykernel_248/2037780541.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Revenue'] = df['Quantity'] * df['Price']
/tmp/ipykernel_248/2037780541.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  d

Data saved


/tmp/ipykernel_248/2037780541.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['MultiItem_Order'] = unique_items_per_invoice.apply(


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,DayOfWeek,TotalRevenue,Is_Most_Popular_Product,OrderSize,CustomerTier,Contains_Gift,TransactionSize,Is_Christmas_Themed,CustomerLoyalty,MultiItem_Order
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,Tuesday,1187.08,False,166,Low,False,Large,True,Loyal,True
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,Tuesday,1187.08,False,166,Low,False,Large,False,Loyal,True
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,Tuesday,1187.08,False,166,Low,False,Large,False,Loyal,True
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,Tuesday,1187.08,False,166,Low,False,Large,False,Loyal,True
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,Tuesday,1187.08,False,166,Low,False,Large,False,Loyal,True


In [31]:
df.to_csv("/content/transformed_retail_data.csv", index=False)
from google.colab import files
files.download("/content/transformed_retail_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>